<a href="https://colab.research.google.com/github/moulyad04/AML23703_1GA23AI026_QUANTUM/blob/main/week07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Requriement **

In [3]:
!pip install -U qiskit-aer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 95.2 MB/s eta 0:00:00


In [4]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print("Qiskit Aer is working!")


Pass: UnrollCustomDefinitions - 0.17643 (ms)

Pass: BasisTranslator - 0.06342 (ms)

Qiskit Aer is working!


Implement the oracle for a constant function f(x)=0 and verify the algorithm outputs '0' (constant). **bold text**

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def constant_oracle():
    """Oracle for f(x) = 0."""
    oracle = QuantumCircuit(2, name="Uf: f(x)=0")
    # f(x) is always 0, so no operation is required.
    return oracle


def deutsch_algorithm(oracle):
    qc = QuantumCircuit(2, 1)

    # Input qubit |0>, ancilla |1>
    qc.x(1)
    qc.h(0)
    qc.h(1)

    # One oracle query
    qc.compose(oracle, inplace=True)

    # Interference
    qc.h(0)
    qc.measure(0, 0)

    return qc


simulator = AerSimulator()

oracle = constant_oracle()
circuit = deutsch_algorithm(oracle)

result = simulator.run(circuit, shots=1024).result()
counts = result.get_counts()

print(counts)


{'0': 1024}


 Implement the oracle for a balanced function f(x)=x and verify the algorithm outputs '1' (balanced). **bold text**

In [6]:
def balanced_oracle():
    """Oracle for f(x) = x."""
    oracle = QuantumCircuit(2, name="Uf: f(x)=x")

    # y -> y XOR x
    oracle.cx(0, 1)

    return oracle


oracle = balanced_oracle()
circuit = deutsch_algorithm(oracle)

result = simulator.run(circuit, shots=1024).result()
counts = result.get_counts()

print(counts)


{'1': 1024}


 Build a generic oracle-selection function that takes a function type as a parameter and dynamically constructs the correct oracle circuit. **bold text**

In [7]:
def build_oracle(function_type):
    """
    Dynamically construct a Deutsch-algorithm oracle.

    function_type:
        "constant_0" -> f(x) = 0
        "constant_1" -> f(x) = 1
        "balanced_x" -> f(x) = x
        "balanced_not_x" -> f(x) = 1 XOR x
    """

    oracle = QuantumCircuit(2, name=f"Oracle: {function_type}")

    if function_type == "constant_0":
        # f(x) = 0
        pass

    elif function_type == "constant_1":
        # f(x) = 1
        oracle.x(1)

    elif function_type == "balanced_x":
        # f(x) = x
        oracle.cx(0, 1)

    elif function_type == "balanced_not_x":
        # f(x) = 1 XOR x
        oracle.x(1)
        oracle.cx(0, 1)

    else:
        raise ValueError(
            "Unknown function type. Choose: "
            "'constant_0', 'constant_1', "
            "'balanced_x', or 'balanced_not_x'."
        )

    return oracle


Extend the demonstration to explain, with a simple slide/diagram, how phase kickback could conceptually apply to a binary classification decision problem.**bold text**

In [8]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# --------------------------------------------------
# Real-world binary classifier
# f(x) = x
#
# x = 0 -> Normal
# x = 1 -> Suspicious
# --------------------------------------------------

def classification_oracle():
    oracle = QuantumCircuit(2, name="Transaction Classifier")

    # f(x) = x
    # If input x = 1, flip the target qubit.
    oracle.cx(0, 1)

    return oracle


# --------------------------------------------------
# Deutsch-style phase kickback demonstration
# --------------------------------------------------

def phase_kickback_classifier():

    qc = QuantumCircuit(2, 1)

    # Input starts as |0>
    #
    # Target starts as |1>
    qc.x(1)

    # Put both qubits into superposition
    qc.h(0)
    qc.h(1)

    # Apply classification oracle
    oracle = classification_oracle()
    qc.compose(oracle, inplace=True)

    # Convert the phase information into
    # a measurable result
    qc.h(0)

    # Measure the input/classification qubit
    qc.measure(0, 0)

    return qc


# --------------------------------------------------
# Run simulation
# --------------------------------------------------

simulator = AerSimulator()

circuit = phase_kickback_classifier()

print("Quantum Circuit:")
print(circuit)

result = simulator.run(
    circuit,
    shots=1024
).result()

counts = result.get_counts()

print("\nMeasurement results:")
print(counts)


# --------------------------------------------------
# Interpret result
# --------------------------------------------------

if counts.get("1", 0) > counts.get("0", 0):
    print("\nResult: 1")
    print("The classifier is BALANCED.")
    print("Class 1 = Suspicious transaction")
else:
    print("\nResult: 0")
    print("The classifier is CONSTANT.")
    print("Class 0 = Normal transaction")


Quantum Circuit:
     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 

Measurement results:
{'1': 1024}

Result: 1
The classifier is BALANCED.
Class 1 = Suspicious transaction
